# Modified TPC_RP Algorithm — CIFAR-10
## Task 3: Algorithm Modification

This notebook implements a **modified** version of TPC_RP for comparison against the original.

**Modification:** [TODO: Describe your modification here]

**Original:** [TODO: Describe what the original does]


---
## 1. Setup: Install Libraries & Imports


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
from torchvision import datasets
from torchvision.models import resnet18

import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.neighbors import NearestNeighbors

import matplotlib.pyplot as plt
import random

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


Using device: cpu


---
## 2. Load CIFAR-10 Dataset


In [8]:

# Load CIFAR-10 training and test sets
basic_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=basic_transform)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=basic_transform)

print(f"Training set size: {len(train_dataset)}")
print(f"Test set size: {len(test_dataset)}")
print(f"Classes: {train_dataset.classes}")


Training set size: 50000
Test set size: 10000
Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


---
## 3. Load Pre-trained SimCLR Features




In [9]:
features = np.load('/content/drive/MyDrive/Colab Notebooks/simclr_features_500epochs.npy')
print(f"Loaded train features: {features.shape}")

train_labels = np.array(train_dataset.targets)
test_labels = np.array(test_dataset.targets)


Loaded train features: (50000, 512)


---
## 4. Typicality Score

[TODO: If your modification changes the typicality calculation, modify the code here.
Otherwise, keep it the same as the original.]


In [10]:
def compute_typicality(features, k_neighbors=20):
    nn_model = NearestNeighbors(n_neighbors=k_neighbors + 1, metric='euclidean')
    nn_model.fit(features)
    distances, _ = nn_model.kneighbors(features)
    distances = distances[:, 1:]
    avg_distances = distances.mean(axis=1)
    typicality = 1.0 / avg_distances
    return typicality


---
## 5. Modified selection function with DNS



In [11]:
# MODIFIED TPC_RP selection with Dynamic Neighbor Scaling (DNS)
# Instead of fixed K=20, K = sqrt(cluster_size) for each cluster

def typicality_clustering_select_dns(features, budget, labeled_indices=None, max_clusters=500):
    if labeled_indices is None:
        labeled_indices = []

    labeled_set = set(labeled_indices)
    n_labeled = len(labeled_indices)
    n_clusters = min(n_labeled + budget, max_clusters)

    print(f"  Running K-Means with {n_clusters} clusters...")
    if n_clusters <= 50:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    else:
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)

    cluster_assignments = kmeans.fit_predict(features)

    cluster_label_counts = {}
    for idx in labeled_indices:
        c = cluster_assignments[idx]
        cluster_label_counts[c] = cluster_label_counts.get(c, 0) + 1

    cluster_to_indices = {}
    for idx in range(len(features)):
        c = cluster_assignments[idx]
        if c not in cluster_to_indices:
            cluster_to_indices[c] = []
        cluster_to_indices[c].append(idx)

    selected = []

    for _ in range(budget):
        best_cluster = None
        best_cluster_fewest_labels = float('inf')
        best_cluster_size = -1

        for c, indices in cluster_to_indices.items():
            if len(indices) <= 5:
                continue
            n_labeled_in_cluster = cluster_label_counts.get(c, 0)
            if (n_labeled_in_cluster < best_cluster_fewest_labels or
                (n_labeled_in_cluster == best_cluster_fewest_labels and
                 len(indices) > best_cluster_size)):
                best_cluster = c
                best_cluster_fewest_labels = n_labeled_in_cluster
                best_cluster_size = len(indices)

        if best_cluster is None:
            break

        cluster_indices = cluster_to_indices[best_cluster]
        cluster_features = features[cluster_indices]

        # DNS: K = sqrt(cluster_size), minimum 2
        k = min(int(np.sqrt(len(cluster_indices))), len(cluster_indices) - 1)
        k = max(k, 2)

        cluster_typicality = compute_typicality(cluster_features, k_neighbors=k)
        sorted_by_typicality = np.argsort(-cluster_typicality)
        for rank in sorted_by_typicality:
            idx = cluster_indices[rank]
            if idx not in labeled_set and idx not in selected:
                selected.append(idx)
                break

        cluster_label_counts[best_cluster] = cluster_label_counts.get(best_cluster, 0) + 1

    return selected

#original for comparison
def typicality_clustering_select(features, budget, labeled_indices=None, max_clusters=500, k_neighbors=20):
    if labeled_indices is None:
        labeled_indices = []

    labeled_set = set(labeled_indices)
    n_labeled = len(labeled_indices)
    n_clusters = min(n_labeled + budget, max_clusters)

    if n_clusters <= 50:
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    else:
        kmeans = MiniBatchKMeans(n_clusters=n_clusters, random_state=42, batch_size=1024)

    cluster_assignments = kmeans.fit_predict(features)

    cluster_label_counts = {}
    for idx in labeled_indices:
        c = cluster_assignments[idx]
        cluster_label_counts[c] = cluster_label_counts.get(c, 0) + 1

    cluster_to_indices = {}
    for idx in range(len(features)):
        c = cluster_assignments[idx]
        if c not in cluster_to_indices:
            cluster_to_indices[c] = []
        cluster_to_indices[c].append(idx)

    selected = []

    for _ in range(budget):
        best_cluster = None
        best_cluster_fewest_labels = float('inf')
        best_cluster_size = -1

        for c, indices in cluster_to_indices.items():
            if len(indices) <= 5:
                continue
            n_labeled_in_cluster = cluster_label_counts.get(c, 0)
            if (n_labeled_in_cluster < best_cluster_fewest_labels or
                (n_labeled_in_cluster == best_cluster_fewest_labels and
                 len(indices) > best_cluster_size)):
                best_cluster = c
                best_cluster_fewest_labels = n_labeled_in_cluster
                best_cluster_size = len(indices)

        if best_cluster is None:
            break

        cluster_indices = cluster_to_indices[best_cluster]
        cluster_features = features[cluster_indices]

        k = min(k_neighbors, len(cluster_indices) - 1)
        if k < 1:
            for idx in cluster_indices:
                if idx not in labeled_set and idx not in selected:
                    selected.append(idx)
                    break
        else:
            cluster_typicality = compute_typicality(cluster_features, k_neighbors=k)
            sorted_by_typicality = np.argsort(-cluster_typicality)
            for rank in sorted_by_typicality:
                idx = cluster_indices[rank]
                if idx not in labeled_set and idx not in selected:
                    selected.append(idx)
                    break

        cluster_label_counts[best_cluster] = cluster_label_counts.get(best_cluster, 0) + 1

    return selected

print("Both selection functions ready (DNS modified + original)")



Both selection functions ready (DNS modified + original)


---
## 6. Framework (i) DNS modified:



In [12]:
# YOUR CODE HERE: Implement classifier training function (same as original)
BUDGET_PER_ROUND = 10
NUM_ROUNDS = 5
NUM_REPETITIONS = 10

mod_fw1_all = []

for rep in range(NUM_REPETITIONS):
    set_seed(rep)
    print(f"\nDNS Framework(i) — Rep {rep+1}/{NUM_REPETITIONS}")
    labeled_indices = []
    round_accs = []

    for round_num in range(NUM_ROUNDS):
        new_indices = typicality_clustering_select_dns(
            features=features, budget=BUDGET_PER_ROUND,
            labeled_indices=labeled_indices, max_clusters=500
        )
        labeled_indices.extend(new_indices)
        acc = train_classifier(labeled_indices, train_dataset, test_dataset, device)
        round_accs.append(acc)
        print(f"  Round {round_num+1}: Budget={len(labeled_indices)}, Acc={acc:.2f}%")

    mod_fw1_all.append(round_accs)

mod_fw1_all = np.array(mod_fw1_all)
mod_fw1_mean = mod_fw1_all.mean(axis=0)
mod_fw1_std = mod_fw1_all.std(axis=0)


DNS Framework(i) — Rep 1/10
  Running K-Means with 10 clusters...


KeyboardInterrupt: 

---
## 7. Active Learning Loop (Modified TPC_RP)

Same setup as original: Budget=10, 5 rounds, 3 repetitions.


In [ ]:
# YOUR CODE HERE: Active learning loop with MODIFIED TPC_RP


---
## 8. Random Baseline for Comparison


In [ ]:
# YOUR CODE HERE: Random baseline (same as original)


---
## 9. Plot Results

Plot the modified TPC_RP vs Random (and optionally vs original TPC_RP).


In [ ]:
# YOUR CODE HERE: Plot modified TPC_RP vs Random results


---
## 10. Comparison: Original vs Modified TPC_RP

Compare the results from the original notebook against this modified version.

[TODO: Enter the original results here for comparison]


In [ ]:
# YOUR CODE HERE: Print comparison table
# original_results = [XX.X, XX.X, XX.X, XX.X, XX.X]  # Copy from original notebook
# modified_results = mean_accuracies  # From this notebook

# print("=" * 75)
# print(f"{'Budget':<10} {'Original TPC_RP':<25} {'Modified TPC_RP':<25} {'Random':<25}")
# print("=" * 75)
# for i in range(NUM_ROUNDS):
#     budget = (i + 1) * BUDGET_PER_ROUND
#     print(f"{budget:<10} {original_results[i]:.2f}%{'':<18} {mean_accuracies[i]:.2f}% +/- {std_accuracies[i]:.2f}%{'':<5} {random_mean[i]:.2f}% +/- {random_std[i]:.2f}%")
# print("=" * 75)


---
## GitHub Repository

🔗 `https://github.com/YOUR_USERNAME/YOUR_REPO_NAME`
